# ARC-AGI-3 scored run — Nine1Eight ARC-color Braille-Sigil Agent

This notebook writes the exact `agent/my_agent.py` used for the Kaggle scored rerun.

Agent SHA256:

```text
b5017c8758e2cf52c794f79cb06f787cc14a567678cd56c20c10e1be9aaced74
```

Core path:

`raw frame -> ARC color matrix 0..9 -> 64x64 padded ARCGrid -> ISO Braille -> object/color stats -> online transition policy`


In [ ]:
# Offline install path used by the ARC-AGI-3 Kaggle Starter dataset.
# Tolerant for local preview; Kaggle scored rerun normally mounts the ARC runtime.
import os, sys, subprocess, pathlib, glob

candidate_wheels = []
for root in ["/kaggle/input", "/kaggle/working", "."]:
    candidate_wheels.extend(glob.glob(os.path.join(root, "**", "arc_agi*.whl"), recursive=True))
    candidate_wheels.extend(glob.glob(os.path.join(root, "**", "arc-agi*.whl"), recursive=True))
    candidate_wheels.extend(glob.glob(os.path.join(root, "**", "arcengine*.whl"), recursive=True))

if candidate_wheels:
    wheel = sorted(candidate_wheels)[0]
    print("Installing ARC runtime wheel:", wheel)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", os.path.dirname(wheel), wheel])
else:
    print("No offline ARC wheel found in preview. Kaggle scored rerun normally provides the runtime.")


In [ ]:
# Write the exact scored agent.
from pathlib import Path
import hashlib

agent_source = '"""\nagent/my_agent.py\n\nNine1Eight Braille-Sigil ARC-AGI-3 agent.\n\nDrop-in target\n--------------\nPlace this file at:\n    ARC-AGI-3-Kaggle-Starter/agent/my_agent.py\n\nIt implements the current ARC-AGI-3 starter contract:\n    class MyAgent(Agent):\n        def is_done(self, frames, latest_frame) -> bool\n        def choose_action(self, frames, latest_frame) -> GameAction\n\nCore idea\n---------\nEvery observed game frame is reduced into a 64x64 binary occupancy map, then\npacked into Unicode Braille using the ISO 11548-1 / drawille-style 2x4 dot cell\nlayout. That gives a compact, deterministic visual state signature:\n\n    raw frame -> background-separated occupancy -> 64x64 -> 32x16 Braille -> hash/stats\n\nThe policy layer uses that signature to build a small online world model:\nstate-action transitions, novelty, progress, no-op penalties, and salience-based\nclick targets. No API calls, no external models, no hard-coded private game\nsolutions, and no dependency beyond the ARC starter runtime.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport math\nimport os\nimport time\nfrom collections import Counter, defaultdict, deque\nfrom dataclasses import dataclass, field\nfrom enum import Enum\nfrom typing import Any, DefaultDict, Dict, Iterable, List, Optional, Sequence, Tuple\n\n# ---------------------------------------------------------------------------\n# ARC runtime imports with safe local-test fallbacks.\n# In the official ARC-AGI-3 runtime these imports resolve to the real classes.\n# ---------------------------------------------------------------------------\ntry:  # ARC-AGI-3-Agents / current starter path\n    from arcengine import FrameData, GameAction, GameState  # type: ignore\nexcept Exception:  # pragma: no cover - fallback exists for local lint/smoke tests\n    class GameState(Enum):\n        NOT_PLAYED = "NOT_PLAYED"\n        PLAYING = "PLAYING"\n        WIN = "WIN"\n        GAME_OVER = "GAME_OVER"\n\n    class _FallbackActionData:\n        def __init__(self) -> None:\n            self.data: Dict[str, Any] = {}\n\n        def model_dump(self) -> Dict[str, Any]:\n            return dict(self.data)\n\n    class GameAction(Enum):\n        RESET = 0\n        ACTION1 = 1\n        ACTION2 = 2\n        ACTION3 = 3\n        ACTION4 = 4\n        ACTION5 = 5\n        ACTION6 = 6\n        ACTION7 = 7\n\n        def is_simple(self) -> bool:\n            return self is not GameAction.ACTION6\n\n        def is_complex(self) -> bool:\n            return self is GameAction.ACTION6\n\n        def set_data(self, data: Dict[str, Any]) -> None:\n            self.action_data = _FallbackActionData()\n            self.action_data.data.update(data)\n\n        @classmethod\n        def from_id(cls, value: int) -> "GameAction":\n            for action in cls:\n                if int(action.value) == int(value):\n                    return action\n            raise ValueError(f"unknown fallback action id: {value}")\n\n    @dataclass\n    class FrameData:  # type: ignore\n        frame: Any = None\n        state: Any = GameState.PLAYING\n        levels_completed: int = 0\n        win_levels: int = 1\n        available_actions: Any = None\n        guid: str = ""\n        game_id: str = "local"\n\ntry:\n    from .agent import Agent  # type: ignore\nexcept Exception:\n    try:\n        from agents.agent import Agent  # type: ignore\n    except Exception:  # pragma: no cover - fallback for isolated import tests\n        class Agent:  # type: ignore\n            MAX_ACTIONS = 80\n\n            def __init__(self, *args: Any, **kwargs: Any) -> None:\n                self.action_counter = 0\n                self.game_id = kwargs.get("game_id", "local")\n\n\n# ---------------------------------------------------------------------------\n# Braille/ISO 11548-1 occupancy grid.\n# ---------------------------------------------------------------------------\nGRID_W = 64\nGRID_H = 64\nCELL_W = 2\nCELL_H = 4\nGLYPH_COLS = GRID_W // CELL_W\nGLYPH_ROWS = GRID_H // CELL_H\nBRAILLE_BASE = 0x2800\n\n# (local_x, local_y) -> bit index, standard 8-dot Braille cell ordering.\n_DOT_BIT: Dict[Tuple[int, int], int] = {\n    (0, 0): 0,  # dot 1\n    (0, 1): 1,  # dot 2\n    (0, 2): 2,  # dot 3\n    (1, 0): 3,  # dot 4\n    (1, 1): 4,  # dot 5\n    (1, 2): 5,  # dot 6\n    (0, 3): 6,  # dot 7\n    (1, 3): 7,  # dot 8\n}\n\n\nclass BrailleGrid64:\n    """64x64 binary occupancy grid with Unicode Braille rendering."""\n\n    __slots__ = ("_cells",)\n\n    def __init__(self) -> None:\n        self._cells = bytearray(GRID_W * GRID_H)\n\n    @classmethod\n    def from_points(cls, points: Iterable[Tuple[int, int]]) -> "BrailleGrid64":\n        grid = cls()\n        for x, y in points:\n            grid.set(x, y, True)\n        return grid\n\n    @classmethod\n    def from_matrix(cls, matrix: Sequence[Sequence[int]]) -> "BrailleGrid64":\n        if len(matrix) != GRID_H:\n            raise ValueError(f"matrix must have {GRID_H} rows, got {len(matrix)}")\n        grid = cls()\n        for y, row in enumerate(matrix):\n            if len(row) != GRID_W:\n                raise ValueError(f"row {y} must have {GRID_W} columns, got {len(row)}")\n            for x, value in enumerate(row):\n                if value:\n                    grid.set(x, y, True)\n        return grid\n\n    def _check_bounds(self, x: int, y: int) -> None:\n        if not (0 <= x < GRID_W and 0 <= y < GRID_H):\n            raise IndexError(f"point ({x}, {y}) out of bounds for {GRID_W}x{GRID_H} grid")\n\n    def set(self, x: int, y: int, val: bool = True) -> None:\n        self._check_bounds(x, y)\n        self._cells[y * GRID_W + x] = 1 if val else 0\n\n    def get(self, x: int, y: int) -> bool:\n        self._check_bounds(x, y)\n        return bool(self._cells[y * GRID_W + x])\n\n    def clear(self) -> None:\n        self._cells = bytearray(GRID_W * GRID_H)\n\n    def points(self) -> List[Tuple[int, int]]:\n        return [\n            (x, y)\n            for y in range(GRID_H)\n            for x in range(GRID_W)\n            if self._cells[y * GRID_W + x]\n        ]\n\n    def to_cell_bytes(self) -> List[List[int]]:\n        out = [[0] * GLYPH_COLS for _ in range(GLYPH_ROWS)]\n        for gy in range(GLYPH_ROWS):\n            base_y = gy * CELL_H\n            for gx in range(GLYPH_COLS):\n                base_x = gx * CELL_W\n                byte_value = 0\n                for (local_x, local_y), bit in _DOT_BIT.items():\n                    index = (base_y + local_y) * GRID_W + (base_x + local_x)\n                    if self._cells[index]:\n                        byte_value |= 1 << bit\n                out[gy][gx] = byte_value\n        return out\n\n    def to_braille_lines(self) -> List[str]:\n        return [\n            "".join(chr(BRAILLE_BASE + byte_value) for byte_value in row)\n            for row in self.to_cell_bytes()\n        ]\n\n    def to_braille_string(self) -> str:\n        return "\\n".join(self.to_braille_lines())\n\n    def _bounding_box(self) -> Optional[Tuple[int, int, int, int]]:\n        pts = self.points()\n        if not pts:\n            return None\n        xs = [point[0] for point in pts]\n        ys = [point[1] for point in pts]\n        return min(xs), min(ys), max(xs), max(ys)\n\n    def _centroid(self) -> Optional[Tuple[float, float]]:\n        pts = self.points()\n        if not pts:\n            return None\n        return (\n            sum(point[0] for point in pts) / len(pts),\n            sum(point[1] for point in pts) / len(pts),\n        )\n\n    def _connected_components(self, connectivity: int = 8) -> List[List[Tuple[int, int]]]:\n        if connectivity not in (4, 8):\n            raise ValueError("connectivity must be 4 or 8")\n\n        if connectivity == 4:\n            neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]\n        else:\n            neighbors = [\n                (-1, -1), (0, -1), (1, -1),\n                (-1, 0),           (1, 0),\n                (-1, 1),  (0, 1),  (1, 1),\n            ]\n\n        seen = bytearray(GRID_W * GRID_H)\n        components: List[List[Tuple[int, int]]] = []\n\n        for y in range(GRID_H):\n            for x in range(GRID_W):\n                index = y * GRID_W + x\n                if not self._cells[index] or seen[index]:\n                    continue\n\n                component: List[Tuple[int, int]] = []\n                queue = deque([(x, y)])\n                seen[index] = 1\n\n                while queue:\n                    current_x, current_y = queue.popleft()\n                    component.append((current_x, current_y))\n                    for dx, dy in neighbors:\n                        nx = current_x + dx\n                        ny = current_y + dy\n                        if 0 <= nx < GRID_W and 0 <= ny < GRID_H:\n                            neighbor_index = ny * GRID_W + nx\n                            if self._cells[neighbor_index] and not seen[neighbor_index]:\n                                seen[neighbor_index] = 1\n                                queue.append((nx, ny))\n\n                components.append(component)\n\n        return components\n\n    def _symmetry(self, bbox: Optional[Tuple[int, int, int, int]]) -> Tuple[bool, bool]:\n        if bbox is None:\n            return False, False\n        x0, y0, x1, y1 = bbox\n        pts = set(self.points())\n        horizontal_mirror = all((x0 + x1 - x, y) in pts for x, y in pts)\n        vertical_mirror = all((x, y0 + y1 - y) in pts for x, y in pts)\n        return horizontal_mirror, vertical_mirror\n\n    def _quadrant_density(self) -> Dict[str, Dict[str, float | int]]:\n        midx = GRID_W // 2\n        midy = GRID_H // 2\n        counts = {"NW": 0, "NE": 0, "SW": 0, "SE": 0}\n        for x, y in self.points():\n            vertical = "N" if y < midy else "S"\n            horizontal = "W" if x < midx else "E"\n            counts[vertical + horizontal] += 1\n        areas = {\n            "NW": midx * midy,\n            "NE": (GRID_W - midx) * midy,\n            "SW": midx * (GRID_H - midy),\n            "SE": (GRID_W - midx) * (GRID_H - midy),\n        }\n        return {\n            quadrant: {\n                "points": counts[quadrant],\n                "density": round(counts[quadrant] / areas[quadrant], 6) if areas[quadrant] else 0.0,\n            }\n            for quadrant in ("NW", "NE", "SW", "SE")\n        }\n\n    def describe(self) -> Dict[str, Any]:\n        pts = self.points()\n        bbox = self._bounding_box()\n        centroid = self._centroid()\n        components_4 = self._connected_components(4)\n        components_8 = self._connected_components(8)\n        horizontal_mirror, vertical_mirror = self._symmetry(bbox)\n        if bbox is None:\n            bbox_info = None\n        else:\n            bbox_info = {\n                "x_min": bbox[0],\n                "y_min": bbox[1],\n                "x_max": bbox[2],\n                "y_max": bbox[3],\n                "width": bbox[2] - bbox[0] + 1,\n                "height": bbox[3] - bbox[1] + 1,\n            }\n        return {\n            "grid_size": [GRID_W, GRID_H],\n            "point_count": len(pts),\n            "density_overall": round(len(pts) / (GRID_W * GRID_H), 6),\n            "bounding_box": bbox_info,\n            "centroid": {"x": round(centroid[0], 3), "y": round(centroid[1], 3)} if centroid else None,\n            "connected_components_4conn": len(components_4),\n            "connected_components_8conn": len(components_8),\n            "largest_component_size": max((len(component) for component in components_8), default=0),\n            "symmetry": {\n                "horizontal_mirror": horizontal_mirror,\n                "vertical_mirror": vertical_mirror,\n            },\n            "quadrant_density": self._quadrant_density(),\n        }\n\n\n@dataclass\nclass BrailleGridAgent:\n    grid: BrailleGrid64 = field(default_factory=BrailleGrid64)\n\n    def load_points(self, points: Iterable[Tuple[int, int]], reset: bool = True) -> "BrailleGridAgent":\n        if reset:\n            self.grid.clear()\n        for x, y in points:\n            self.grid.set(x, y, True)\n        return self\n\n    def load_matrix(self, matrix: Sequence[Sequence[int]]) -> "BrailleGridAgent":\n        self.grid = BrailleGrid64.from_matrix(matrix)\n        return self\n\n    def describe_json(self) -> Dict[str, Any]:\n        out = self.grid.describe()\n        out["braille"] = self.grid.to_braille_lines()\n        return out\n\n\nCOLOR_NAMES: Dict[int, str] = {\n    0: "black/empty",\n    1: "blue",\n    2: "red",\n    3: "green",\n    4: "yellow",\n    5: "gray",\n    6: "magenta",\n    7: "orange",\n    8: "cyan",\n    9: "pink",\n}\n\n\ndef _validate_color(value: int) -> int:\n    ivalue = int(value)\n    if not 0 <= ivalue <= 9:\n        raise ValueError(f"ARC color must be in 0..9, got {value!r}")\n    return ivalue\n\n\ndef _bbox(points: Sequence[Tuple[int, int]]) -> Optional[Dict[str, int]]:\n    if not points:\n        return None\n    xs = [p[0] for p in points]\n    ys = [p[1] for p in points]\n    return {\n        "x_min": min(xs),\n        "y_min": min(ys),\n        "x_max": max(xs),\n        "y_max": max(ys),\n        "width": max(xs) - min(xs) + 1,\n        "height": max(ys) - min(ys) + 1,\n    }\n\n\ndef _centroid(points: Sequence[Tuple[int, int]]) -> Optional[Dict[str, float]]:\n    if not points:\n        return None\n    return {\n        "x": round(sum(x for x, _ in points) / len(points), 3),\n        "y": round(sum(y for _, y in points) / len(points), 3),\n    }\n\n\nclass ARCGrid:\n    """Multi-color ARC grid backed by a 64x64 bytearray.\n\n    `width` and `height` retain the logical task/frame size. Storage remains fixed\n    at 64x64 so Braille rendering is deterministic and no padding branch is needed.\n    """\n\n    __slots__ = ("_cells", "width", "height")\n\n    def __init__(self, width: int = GRID_W, height: int = GRID_H) -> None:\n        if not (1 <= int(width) <= GRID_W and 1 <= int(height) <= GRID_H):\n            raise ValueError(f"ARCGrid logical size must fit within {GRID_W}x{GRID_H}, got {width}x{height}")\n        self.width = int(width)\n        self.height = int(height)\n        self._cells = bytearray(GRID_W * GRID_H)\n\n    @classmethod\n    def from_matrix(cls, matrix: Sequence[Sequence[int]]) -> "ARCGrid":\n        if not matrix:\n            return cls(1, 1)\n        height = len(matrix)\n        width = max((len(row) for row in matrix), default=0)\n        if width <= 0:\n            return cls(1, 1)\n        if width > GRID_W or height > GRID_H:\n            matrix = cls._resample_matrix(matrix, GRID_W, GRID_H)\n            height, width = GRID_H, GRID_W\n        grid = cls(width, height)\n        for y, row in enumerate(matrix):\n            for x in range(width):\n                value = row[x] if x < len(row) else 0\n                grid.set(x, y, _validate_color(value))\n        return grid\n\n    @staticmethod\n    def _resample_matrix(matrix: Sequence[Sequence[int]], target_w: int, target_h: int) -> List[List[int]]:\n        src_h = len(matrix)\n        src_w = max((len(row) for row in matrix), default=1)\n        out: List[List[int]] = []\n        for ty in range(target_h):\n            sy = min(src_h - 1, max(0, int((ty + 0.5) * src_h / target_h)))\n            row_out: List[int] = []\n            row = matrix[sy] if sy < len(matrix) else []\n            for tx in range(target_w):\n                sx = min(src_w - 1, max(0, int((tx + 0.5) * src_w / target_w)))\n                row_out.append(_validate_color(row[sx] if sx < len(row) else 0))\n            out.append(row_out)\n        return out\n\n    def clone(self) -> "ARCGrid":\n        copied = ARCGrid(self.width, self.height)\n        copied._cells[:] = self._cells\n        return copied\n\n    def set(self, x: int, y: int, value: int) -> None:\n        if not (0 <= int(x) < self.width and 0 <= int(y) < self.height):\n            raise IndexError(f"point ({x}, {y}) out of bounds for {self.width}x{self.height}")\n        self._cells[int(y) * GRID_W + int(x)] = _validate_color(value)\n\n    def get(self, x: int, y: int) -> int:\n        if not (0 <= int(x) < self.width and 0 <= int(y) < self.height):\n            return 0\n        return int(self._cells[int(y) * GRID_W + int(x)])\n\n    def to_matrix(self) -> List[List[int]]:\n        return [[self.get(x, y) for x in range(self.width)] for y in range(self.height)]\n\n    def points(self, include_zero: bool = False) -> List[Tuple[int, int, int]]:\n        pts: List[Tuple[int, int, int]] = []\n        for y in range(self.height):\n            for x in range(self.width):\n                color = self.get(x, y)\n                if include_zero or color != 0:\n                    pts.append((x, y, color))\n        return pts\n\n    def occupied_points(self) -> List[Tuple[int, int]]:\n        return [(x, y) for x, y, color in self.points(False) if color != 0]\n\n    def color_histogram(self) -> Dict[int, int]:\n        counts: Dict[int, int] = {color: 0 for color in range(10)}\n        for _, _, color in self.points(include_zero=True):\n            counts[color] += 1\n        return {color: count for color, count in counts.items() if count > 0}\n\n    def to_braille_lines(self) -> List[str]:\n        out = [[0] * GLYPH_COLS for _ in range(GLYPH_ROWS)]\n        for gy in range(GLYPH_ROWS):\n            base_y = gy * CELL_H\n            for gx in range(GLYPH_COLS):\n                base_x = gx * CELL_W\n                byte_value = 0\n                for (lx, ly), bit in _DOT_BIT.items():\n                    if self.get(base_x + lx, base_y + ly) != 0:\n                        byte_value |= 1 << bit\n                out[gy][gx] = byte_value\n        return ["".join(chr(BRAILLE_BASE + value) for value in row) for row in out]\n\n    def to_braille_string(self) -> str:\n        return "\\n".join(self.to_braille_lines())\n\n    def render_colored_legend(self) -> str:\n        present = self.color_histogram()\n        return "Colors: " + ", ".join(f"{color}:{COLOR_NAMES[color]}" for color in sorted(present) if color != 0)\n\n    def render_full(self) -> str:\n        return "```\\n" + self.to_braille_string() + "\\n```\\n" + self.render_colored_legend()\n\n    def connected_components(self, color: Optional[int] = None, connectivity: int = 8) -> List[List[Tuple[int, int]]]:\n        if connectivity not in (4, 8):\n            raise ValueError("connectivity must be 4 or 8")\n        if color is not None:\n            color = _validate_color(color)\n        neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]\n        if connectivity == 8:\n            neighbors = [(-1, -1), (0, -1), (1, -1), (-1, 0), (1, 0), (-1, 1), (0, 1), (1, 1)]\n        seen = bytearray(GRID_W * GRID_H)\n        components: List[List[Tuple[int, int]]] = []\n        for y in range(self.height):\n            for x in range(self.width):\n                current = self.get(x, y)\n                if current == 0 or seen[y * GRID_W + x] or (color is not None and current != color):\n                    continue\n                comp: List[Tuple[int, int]] = []\n                q: deque[Tuple[int, int]] = deque([(x, y)])\n                seen[y * GRID_W + x] = 1\n                while q:\n                    cx, cy = q.popleft()\n                    comp.append((cx, cy))\n                    for dx, dy in neighbors:\n                        nx, ny = cx + dx, cy + dy\n                        if 0 <= nx < self.width and 0 <= ny < self.height:\n                            idx = ny * GRID_W + nx\n                            if not seen[idx] and self.get(nx, ny) == current:\n                                seen[idx] = 1\n                                q.append((nx, ny))\n                components.append(comp)\n        return components\n\n    def objects_by_color(self, connectivity: int = 8) -> Dict[int, List[List[Tuple[int, int]]]]:\n        result: DefaultDict[int, List[List[Tuple[int, int]]]] = defaultdict(list)\n        for comp in self.connected_components(None, connectivity):\n            if comp:\n                result[self.get(comp[0][0], comp[0][1])].append(comp)\n        return dict(result)\n\n    def color_bounding_boxes(self) -> Dict[int, Optional[Dict[str, int]]]:\n        boxes: Dict[int, Optional[Dict[str, int]]] = {}\n        for color in sorted(self.objects_by_color().keys()):\n            color_points = [(x, y) for x, y, c in self.points(False) if c == color]\n            boxes[color] = _bbox(color_points)\n        return boxes\n\n    def quadrant_density(self) -> Dict[str, Dict[str, float | int]]:\n        midx = max(1, self.width // 2)\n        midy = max(1, self.height // 2)\n        counts = {"NW": 0, "NE": 0, "SW": 0, "SE": 0}\n        areas = {\n            "NW": midx * midy,\n            "NE": (self.width - midx) * midy,\n            "SW": midx * (self.height - midy),\n            "SE": (self.width - midx) * (self.height - midy),\n        }\n        for x, y in self.occupied_points():\n            key = ("N" if y < midy else "S") + ("W" if x < midx else "E")\n            counts[key] += 1\n        return {q: {"points": counts[q], "density": round(counts[q] / areas[q], 6) if areas[q] else 0.0} for q in counts}\n\n    def symmetry(self) -> Dict[str, bool]:\n        pts_by_color = {(x, y): color for x, y, color in self.points(False)}\n        occupied = [(x, y) for x, y, _ in self.points(False)]\n        box = _bbox(occupied)\n        if box is None:\n            return {"horizontal_mirror": False, "vertical_mirror": False, "main_diagonal": False, "anti_diagonal": False}\n        x0, x1 = box["x_min"], box["x_max"]\n        y0, y1 = box["y_min"], box["y_max"]\n        horizontal = all(pts_by_color.get((x0 + x1 - x, y)) == color for (x, y), color in pts_by_color.items())\n        vertical = all(pts_by_color.get((x, y0 + y1 - y)) == color for (x, y), color in pts_by_color.items())\n        main_diag = box["width"] == box["height"] and all(\n            pts_by_color.get((x0 + (y - y0), y0 + (x - x0))) == color for (x, y), color in pts_by_color.items()\n        )\n        anti_diag = box["width"] == box["height"] and all(\n            pts_by_color.get((x0 + (y1 - y), y0 + (x1 - x))) == color for (x, y), color in pts_by_color.items()\n        )\n        return {"horizontal_mirror": horizontal, "vertical_mirror": vertical, "main_diagonal": main_diag, "anti_diagonal": anti_diag}\n\n    def crop_to_bbox(self, bbox: Optional[Dict[str, int]] = None) -> "ARCGrid":\n        bbox = bbox or _bbox(self.occupied_points())\n        if bbox is None:\n            return ARCGrid(1, 1)\n        matrix = [\n            [self.get(x, y) for x in range(bbox["x_min"], bbox["x_max"] + 1)]\n            for y in range(bbox["y_min"], bbox["y_max"] + 1)\n        ]\n        return ARCGrid.from_matrix(matrix)\n\n    def rotate90(self, turns: int = 1) -> "ARCGrid":\n        turns = turns % 4\n        matrix = self.to_matrix()\n        for _ in range(turns):\n            matrix = [list(row) for row in zip(*matrix[::-1])]\n        return ARCGrid.from_matrix(matrix)\n\n    def flip_horizontal(self) -> "ARCGrid":\n        return ARCGrid.from_matrix([list(reversed(row)) for row in self.to_matrix()])\n\n    def flip_vertical(self) -> "ARCGrid":\n        return ARCGrid.from_matrix(list(reversed(self.to_matrix())))\n\n    def translate(self, dx: int, dy: int, background: int = 0) -> "ARCGrid":\n        out = ARCGrid(self.width, self.height)\n        for y in range(self.height):\n            for x in range(self.width):\n                out.set(x, y, background)\n        for x, y, color in self.points(False):\n            nx, ny = x + int(dx), y + int(dy)\n            if 0 <= nx < self.width and 0 <= ny < self.height:\n                out.set(nx, ny, color)\n        return out\n\n    def describe(self) -> Dict[str, Any]:\n        objects = self.objects_by_color(connectivity=8)\n        occupied = self.occupied_points()\n        object_summaries: Dict[int, List[Dict[str, Any]]] = {}\n        for color, comps in sorted(objects.items()):\n            object_summaries[color] = [\n                {\n                    "size": len(comp),\n                    "bbox": _bbox(comp),\n                    "centroid": _centroid(comp),\n                }\n                for comp in comps\n            ]\n        return {\n            "grid_size": [self.width, self.height],\n            "total_colored_cells": len(occupied),\n            "density_overall": round(len(occupied) / max(1, self.width * self.height), 6),\n            "colors_present": sorted(objects.keys()),\n            "color_histogram": self.color_histogram(),\n            "objects_per_color": {color: len(comps) for color, comps in sorted(objects.items())},\n            "largest_object_size": max((len(comp) for comps in objects.values() for comp in comps), default=0),\n            "bounding_box": _bbox(occupied),\n            "centroid": _centroid(occupied),\n            "color_bounding_boxes": self.color_bounding_boxes(),\n            "quadrant_density": self.quadrant_density(),\n            "symmetry": self.symmetry(),\n            "objects": object_summaries,\n            "braille": self.to_braille_lines(),\n        }\n\n\n@dataclass\nclass ARCGridAgent:\n    """Stateful wrapper for ARC task, train/test pair, or single matrix analysis."""\n\n    grid: ARCGrid = field(default_factory=ARCGrid)\n\n    def load_task_pair(self, pair: Dict[str, Any], key: str = "input") -> "ARCGridAgent":\n        if key not in pair:\n            raise KeyError(f"task pair missing key {key!r}")\n        self.grid = ARCGrid.from_matrix(pair[key])\n        return self\n\n    def load_matrix(self, matrix: Sequence[Sequence[int]]) -> "ARCGridAgent":\n        self.grid = ARCGrid.from_matrix(matrix)\n        return self\n\n    def describe(self) -> str:\n        return self.grid.render_full() + "\\n\\n" + json.dumps(self.grid.describe(), indent=2)\n\n    def describe_json(self) -> Dict[str, Any]:\n        return self.grid.describe()\n\n    def get_objects(self) -> Dict[int, List[List[Tuple[int, int]]]]:\n        return self.grid.objects_by_color()\n\n\ndef load_task(path: str) -> Dict[str, Any]:\n    with open(path, "r", encoding="utf-8") as handle:\n        return json.load(handle)\n\n\ndef analyze_task(task: Dict[str, Any]) -> Dict[str, Any]:\n    out: Dict[str, Any] = {"train": [], "test": []}\n    for split in ("train", "test"):\n        for index, pair in enumerate(task.get(split, [])):\n            row: Dict[str, Any] = {"index": index}\n            if "input" in pair:\n                row["input"] = ARCGridAgent().load_task_pair(pair, "input").describe_json()\n            if "output" in pair:\n                row["output"] = ARCGridAgent().load_task_pair(pair, "output").describe_json()\n            out[split].append(row)\n    return out\n\n# ---------------------------------------------------------------------------\n# Visual signatures and online transition memory.\n# ---------------------------------------------------------------------------\n@dataclass(frozen=True)\nclass VisualSignature:\n    key: str\n    raw_hash: str\n    braille_hash: str\n    braille: Tuple[str, ...]\n    point_count: int\n    density: float\n    bbox: Optional[Dict[str, int]]\n    centroid: Optional[Tuple[float, float]]\n    quadrant_density: Dict[str, Dict[str, float | int]]\n    color_histogram: Tuple[Tuple[str, int], ...]\n    changed_points: Tuple[Tuple[int, int], ...]\n    changed_count: int\n    changed_centroid: Optional[Tuple[float, float]]\n\n\n@dataclass\nclass ActionStats:\n    tries: int = 0\n    progress: int = 0\n    regressions: int = 0\n    wins: int = 0\n    noops: int = 0\n    novelty: int = 0\n    changed_total: int = 0\n\n    def mean_change(self) -> float:\n        return self.changed_total / self.tries if self.tries else 0.0\n\n    def score_prior(self) -> float:\n        return (\n            self.progress * 20.0\n            + self.wins * 100.0\n            + self.novelty * 5.0\n            + self.mean_change() * 0.05\n            - self.noops * 8.0\n            - self.regressions * 20.0\n        )\n\n\n@dataclass\nclass TransitionRecord:\n    source_key: str\n    action_id: int\n    target_key: str\n    before_levels: int\n    after_levels: int\n    changed_count: int\n    timestamp: float\n\n\nclass OnlineWorldModel:\n    """Small deterministic model of observed state/action outcomes."""\n\n    def __init__(self) -> None:\n        self.state_visits: DefaultDict[str, int] = defaultdict(int)\n        self.state_action_stats: DefaultDict[Tuple[str, int], ActionStats] = defaultdict(ActionStats)\n        self.global_action_stats: DefaultDict[int, ActionStats] = defaultdict(ActionStats)\n        self.transitions: Dict[Tuple[str, int], TransitionRecord] = {}\n        self.known_states: set[str] = set()\n        self.last_level_count: int = 0\n\n    def observe_state(self, state_key: str) -> None:\n        self.state_visits[state_key] += 1\n        self.known_states.add(state_key)\n\n    def record_transition(\n        self,\n        source_key: str,\n        action_id: int,\n        target_key: str,\n        before_levels: int,\n        after_levels: int,\n        changed_count: int,\n        is_win: bool,\n    ) -> None:\n        state_stats = self.state_action_stats[(source_key, action_id)]\n        global_stats = self.global_action_stats[action_id]\n        for stats in (state_stats, global_stats):\n            stats.tries += 1\n            stats.changed_total += changed_count\n            if after_levels > before_levels:\n                stats.progress += after_levels - before_levels\n            if after_levels < before_levels:\n                stats.regressions += before_levels - after_levels\n            if is_win:\n                stats.wins += 1\n            if target_key == source_key or changed_count == 0:\n                stats.noops += 1\n            if target_key not in self.known_states:\n                stats.novelty += 1\n        self.transitions[(source_key, action_id)] = TransitionRecord(\n            source_key=source_key,\n            action_id=action_id,\n            target_key=target_key,\n            before_levels=before_levels,\n            after_levels=after_levels,\n            changed_count=changed_count,\n            timestamp=time.time(),\n        )\n        self.known_states.add(target_key)\n        self.last_level_count = max(self.last_level_count, after_levels)\n\n\n# ---------------------------------------------------------------------------\n# Main ARC agent.\n# ---------------------------------------------------------------------------\nclass MyAgent(Agent):\n    """ARC-color Braille-sigil ARC-AGI-3 agent with online exploration/world-model policy."""\n\n    MAX_ACTIONS = int(os.getenv("NINE18_MAX_ACTIONS", "80"))\n\n    def __init__(self, *args: Any, **kwargs: Any) -> None:\n        super().__init__(*args, **kwargs)\n        self.world = OnlineWorldModel()\n        self.last_visual: Optional[VisualSignature] = None\n        self.last_state_key: Optional[str] = None\n        self.last_levels_completed: int = 0\n        self.pending_action_id: Optional[int] = None\n        self.pending_action_key: Optional[str] = None\n        self.pending_before_levels: int = 0\n        self.step_index: int = 0\n        self.trace_enabled = os.getenv("NINE18_TRACE", "0").strip().lower() in {"1", "true", "yes"}\n        self.trace_path = os.getenv("NINE18_TRACE_PATH", "nine18_braille_agent_trace.jsonl")\n\n    @property\n    def name(self) -> str:\n        base_name = super().name if hasattr(super(), "name") else self.__class__.__name__.lower()\n        return f"{base_name}.arc_color_braille_sigil_v2"\n\n    def is_done(self, frames: List[FrameData], latest_frame: FrameData) -> bool:\n        state = getattr(latest_frame, "state", None)\n        levels_completed = int(getattr(latest_frame, "levels_completed", 0) or 0)\n        win_levels = int(getattr(latest_frame, "win_levels", 0) or 0)\n        return any(\n            [\n                self._state_is(state, "WIN"),\n                bool(win_levels and levels_completed >= win_levels),\n                bool(getattr(self, "action_counter", 0) >= self.MAX_ACTIONS),\n            ]\n        )\n\n    def choose_action(self, frames: List[FrameData], latest_frame: FrameData) -> GameAction:\n        self.step_index += 1\n        visual = self._visual_signature(latest_frame)\n        state = getattr(latest_frame, "state", None)\n        current_levels = int(getattr(latest_frame, "levels_completed", 0) or 0)\n\n        self._commit_pending_transition(visual, state, current_levels)\n        self.world.observe_state(visual.key)\n\n        if self._state_is(state, "NOT_PLAYED") or self._state_is(state, "GAME_OVER"):\n            action = GameAction.RESET\n            self._attach_reasoning(\n                action,\n                {\n                    "agent": "nine18_arc_color_braille_sigil_v2",\n                    "mode": "reset_required",\n                    "state": self._state_name(state),\n                    "braille_hash": visual.braille_hash,\n                },\n            )\n            self._set_pending(action, visual, current_levels)\n            self._trace(latest_frame, visual, action, "reset_required", [])\n            return action\n\n        candidates = self._available_actions(latest_frame)\n        if not candidates:\n            candidates = self._all_non_reset_actions()\n        candidates = [action for action in candidates if not self._action_is_reset(action)]\n        if not candidates:\n            action = GameAction.RESET\n            self._attach_reasoning(action, {"agent": "nine18_arc_color_braille_sigil_v2", "mode": "fallback_reset_no_candidates"})\n            self._set_pending(action, visual, current_levels)\n            self._trace(latest_frame, visual, action, "fallback_reset_no_candidates", [])\n            return action\n\n        scored = [(self._score_action(action, visual, current_levels), action) for action in candidates]\n        scored.sort(key=lambda item: item[0], reverse=True)\n        chosen = scored[0][1]\n\n        if self._is_complex_action(chosen):\n            target_x, target_y, target_source = self._choose_target(visual)\n            self._set_action_data(chosen, {"x": int(target_x), "y": int(target_y)})\n        else:\n            target_source = "simple_action_no_coordinates"\n\n        action_id = self._action_id(chosen)\n        reasoning = {\n            "agent": "nine18_arc_color_braille_sigil_v2",\n            "mode": "braille_world_model_ucb",\n            "step": self.step_index,\n            "state": self._state_name(state),\n            "levels_completed": current_levels,\n            "known_states": len(self.world.known_states),\n            "available_action_ids": [self._action_id(action) for action in candidates],\n            "selected_action_id": action_id,\n            "selected_action_name": self._action_name(chosen),\n            "visual": {\n                "raw_hash": visual.raw_hash,\n                "braille_hash": visual.braille_hash,\n                "point_count": visual.point_count,\n                "density": visual.density,\n                "bbox": visual.bbox,\n                "centroid": visual.centroid,\n                "changed_count": visual.changed_count,\n                "changed_centroid": visual.changed_centroid,\n            },\n            "target_source": target_source,\n            "scoreboard": [\n                {\n                    "action_id": self._action_id(action),\n                    "action_name": self._action_name(action),\n                    "score": round(score, 5),\n                    "tries_here": self.world.state_action_stats[(visual.key, self._action_id(action))].tries,\n                    "global_tries": self.world.global_action_stats[self._action_id(action)].tries,\n                }\n                for score, action in scored[: min(7, len(scored))]\n            ],\n        }\n        self._attach_reasoning(chosen, reasoning)\n        self._set_pending(chosen, visual, current_levels)\n        self._trace(latest_frame, visual, chosen, "selected", scored)\n        return chosen\n\n    # ------------------------------------------------------------------\n    # Visual processing.\n    # ------------------------------------------------------------------\n    def _visual_signature(self, latest_frame: FrameData) -> VisualSignature:\n        matrix = self._coerce_frame_matrix(getattr(latest_frame, "frame", None))\n        raw_hash = self._hash_jsonable(matrix)\n\n        arc_matrix = self._matrix_to_arc_color_matrix(matrix)\n        arc_grid = ARCGrid.from_matrix(arc_matrix) if arc_matrix else ARCGrid(1, 1)\n        description = arc_grid.describe()\n        braille_lines = tuple(description["braille"])\n        braille_hash = hashlib.sha256("\\n".join(braille_lines).encode("utf-8")).hexdigest()[:24]\n\n        occupied_points = tuple(sorted(arc_grid.occupied_points()))\n        point_set = set(occupied_points)\n        if self.last_visual is None:\n            changed_points = occupied_points\n        else:\n            previous_points = self._points_from_braille(self.last_visual.braille)\n            changed_points = tuple(sorted(point_set.symmetric_difference(previous_points)))\n\n        changed_centroid = self._centroid_tuple(changed_points)\n        centroid_dict = description.get("centroid")\n        centroid = None\n        if centroid_dict is not None:\n            centroid = (float(centroid_dict["x"]), float(centroid_dict["y"]))\n\n        histogram_dict = description.get("color_histogram", {})\n        histogram = tuple((str(k), int(v)) for k, v in sorted(histogram_dict.items(), key=lambda item: int(item[0])))\n\n        state_key_seed = {\n            "raw_hash": raw_hash,\n            "braille_hash": braille_hash,\n            "levels": int(getattr(latest_frame, "levels_completed", 0) or 0),\n            "state": self._state_name(getattr(latest_frame, "state", None)),\n            "colors_present": description.get("colors_present", []),\n            "objects_per_color": description.get("objects_per_color", {}),\n            "symmetry": description.get("symmetry", {}),\n        }\n        key = hashlib.sha256(json.dumps(state_key_seed, sort_keys=True, default=str).encode("utf-8")).hexdigest()[:32]\n        return VisualSignature(\n            key=key,\n            raw_hash=raw_hash,\n            braille_hash=braille_hash,\n            braille=braille_lines,\n            point_count=int(description["total_colored_cells"]),\n            density=float(description["density_overall"]),\n            bbox=description.get("bounding_box"),\n            centroid=centroid,\n            quadrant_density=description.get("quadrant_density", {}),\n            color_histogram=histogram,\n            changed_points=changed_points,\n            changed_count=len(changed_points),\n            changed_centroid=changed_centroid,\n        )\n\n    def _coerce_frame_matrix(self, frame: Any) -> List[List[Any]]:\n        if frame is None:\n            return []\n        if hasattr(frame, "tolist"):\n            frame = frame.tolist()\n        if not isinstance(frame, list):\n            return [[self._freeze_cell(frame)]]\n        if len(frame) == 0:\n            return []\n\n        # Normalize 1D frames into a single row; normalize ndarray-like rows.\n        if frame and not isinstance(frame[0], (list, tuple)) and not hasattr(frame[0], "tolist"):\n            return [[self._freeze_cell(cell) for cell in frame]]\n\n        matrix: List[List[Any]] = []\n        for row in frame:\n            if hasattr(row, "tolist"):\n                row = row.tolist()\n            if not isinstance(row, (list, tuple)):\n                matrix.append([self._freeze_cell(row)])\n                continue\n            matrix.append([self._freeze_cell(cell) for cell in row])\n        return matrix\n\n    def _freeze_cell(self, value: Any) -> Any:\n        if hasattr(value, "tolist"):\n            value = value.tolist()\n        if isinstance(value, (list, tuple)):\n            return tuple(self._freeze_cell(item) for item in value)\n        if isinstance(value, dict):\n            return tuple(sorted((str(key), self._freeze_cell(val)) for key, val in value.items()))\n        if isinstance(value, (int, float, str, bool)) or value is None:\n            return value\n        return repr(value)\n\n    def _matrix_to_arc_color_matrix(self, matrix: List[List[Any]]) -> List[List[int]]:\n        """Convert an arbitrary ARC-AGI-3 frame matrix into a 0..9 ARC color matrix.\n\n        Native integer colors 0..9 are preserved. RGB/tuple/object cells are collapsed\n        by deterministic token mapping after treating the most common cell as background 0.\n        Frames larger than 64x64 are nearest-neighbor downsampled by ARCGrid.from_matrix.\n        """\n        if not matrix:\n            return []\n        height = len(matrix)\n        width = max((len(row) for row in matrix), default=0)\n        if width <= 0:\n            return []\n\n        flat: List[Any] = []\n        for row in matrix:\n            flat.extend(row)\n        background = Counter(flat).most_common(1)[0][0] if flat else 0\n\n        token_to_color: Dict[str, int] = {}\n        out: List[List[int]] = []\n        for row in matrix:\n            out_row: List[int] = []\n            for cell in row:\n                if cell == background or cell in (None, False, "", "0"):\n                    out_row.append(0)\n                    continue\n                if isinstance(cell, (int, float)) and int(cell) == cell and 0 <= int(cell) <= 9:\n                    out_row.append(int(cell))\n                    continue\n                token = self._cell_to_token(cell)\n                if token not in token_to_color:\n                    digest = int(hashlib.sha256(token.encode("utf-8")).hexdigest()[:8], 16)\n                    candidate = (digest % 9) + 1\n                    if candidate in token_to_color.values() and len(set(token_to_color.values())) < 9:\n                        for color in range(1, 10):\n                            if color not in token_to_color.values():\n                                candidate = color\n                                break\n                    token_to_color[token] = candidate\n                out_row.append(token_to_color[token])\n            if len(out_row) < width:\n                out_row.extend([0] * (width - len(out_row)))\n            out.append(out_row)\n        return out\n\n    def _matrix_to_64_points(self, matrix: List[List[Any]]) -> Tuple[List[Tuple[int, int]], Tuple[Tuple[str, int], ...]]:\n        if not matrix:\n            return [], tuple()\n        height = len(matrix)\n        width = max((len(row) for row in matrix), default=0)\n        if width <= 0:\n            return [], tuple()\n\n        flat: List[Any] = []\n        for row in matrix:\n            flat.extend(row)\n        background = Counter(flat).most_common(1)[0][0] if flat else 0\n        histogram_counter = Counter(self._cell_to_token(cell) for cell in flat)\n        histogram = tuple(histogram_counter.most_common(12))\n\n        points: set[Tuple[int, int]] = set()\n        for y, row in enumerate(matrix):\n            if not row:\n                continue\n            row_width = len(row)\n            for x, cell in enumerate(row):\n                if self._is_occupied_cell(cell, background):\n                    mapped_x = min(GRID_W - 1, max(0, int((x + 0.5) * GRID_W / max(1, row_width))))\n                    mapped_y = min(GRID_H - 1, max(0, int((y + 0.5) * GRID_H / max(1, height))))\n                    points.add((mapped_x, mapped_y))\n        return sorted(points), histogram\n\n    def _is_occupied_cell(self, cell: Any, background: Any) -> bool:\n        if cell == background:\n            return False\n        if cell in (None, False, 0, "", "0"):\n            return False\n        if isinstance(cell, tuple) and all(item in (0, 0.0, False, None) for item in cell):\n            return False\n        return True\n\n    def _cell_to_token(self, cell: Any) -> str:\n        if isinstance(cell, tuple):\n            return "[" + ",".join(self._cell_to_token(item) for item in cell) + "]"\n        return str(cell)\n\n    def _points_from_braille(self, braille_lines: Tuple[str, ...]) -> set[Tuple[int, int]]:\n        points: set[Tuple[int, int]] = set()\n        for gy, line in enumerate(braille_lines):\n            for gx, ch in enumerate(line):\n                value = ord(ch) - BRAILLE_BASE\n                if value < 0 or value > 255:\n                    continue\n                for (local_x, local_y), bit in _DOT_BIT.items():\n                    if value & (1 << bit):\n                        points.add((gx * CELL_W + local_x, gy * CELL_H + local_y))\n        return points\n\n    # ------------------------------------------------------------------\n    # World-model policy.\n    # ------------------------------------------------------------------\n    def _commit_pending_transition(self, visual: VisualSignature, state: Any, current_levels: int) -> None:\n        if self.pending_action_id is None or self.pending_action_key is None:\n            self.last_visual = visual\n            self.last_state_key = visual.key\n            self.last_levels_completed = current_levels\n            return\n        self.world.record_transition(\n            source_key=self.pending_action_key,\n            action_id=self.pending_action_id,\n            target_key=visual.key,\n            before_levels=self.pending_before_levels,\n            after_levels=current_levels,\n            changed_count=visual.changed_count,\n            is_win=self._state_is(state, "WIN"),\n        )\n        self.pending_action_id = None\n        self.pending_action_key = None\n        self.pending_before_levels = 0\n        self.last_visual = visual\n        self.last_state_key = visual.key\n        self.last_levels_completed = current_levels\n\n    def _score_action(self, action: GameAction, visual: VisualSignature, current_levels: int) -> float:\n        action_id = self._action_id(action)\n        state_stats = self.world.state_action_stats[(visual.key, action_id)]\n        global_stats = self.world.global_action_stats[action_id]\n        visit_count = max(1, self.world.state_visits[visual.key])\n        tries_here = state_stats.tries\n\n        # UCB term: aggressive on untried state-action pairs, calmer after testing.\n        if tries_here == 0:\n            exploration = 60.0\n        else:\n            exploration = 12.0 * math.sqrt(math.log(visit_count + 2.0) / (tries_here + 1.0))\n\n        predicted = 0.0\n        transition = self.world.transitions.get((visual.key, action_id))\n        if transition is not None:\n            level_delta = transition.after_levels - transition.before_levels\n            predicted += level_delta * 35.0\n            predicted += min(25.0, transition.changed_count * 0.05)\n            if transition.target_key == visual.key:\n                predicted -= 18.0\n\n        global_prior = global_stats.score_prior() * 0.35\n        state_prior = state_stats.score_prior() * 0.8\n        novelty_pressure = max(0.0, 25.0 - self.world.state_visits[visual.key] * 3.0)\n        salience_bonus = 0.0\n        if self._is_complex_action(action):\n            salience_bonus += 8.0 if visual.bbox or visual.changed_centroid else -4.0\n        else:\n            salience_bonus += self._deterministic_action_phase_bonus(action_id, visual.key)\n\n        loop_penalty = max(0.0, tries_here - 1.0) * 7.5\n        game_progress_bias = current_levels * 0.25\n        tie_break = self._stable_float(f"{visual.key}:{action_id}:{self.step_index}") * 0.01\n        return (\n            exploration\n            + predicted\n            + global_prior\n            + state_prior\n            + novelty_pressure\n            + salience_bonus\n            + game_progress_bias\n            + tie_break\n            - loop_penalty\n        )\n\n    def _deterministic_action_phase_bonus(self, action_id: int, state_key: str) -> float:\n        # Keeps exploration from becoming ACTION1-only when every action is new.\n        seed = int(hashlib.sha256(f"{state_key}:{self.step_index}".encode("utf-8")).hexdigest()[:8], 16)\n        phase = seed % 7\n        return 3.0 if action_id == (phase + 1) else 0.0\n\n    def _choose_target(self, visual: VisualSignature) -> Tuple[int, int, str]:\n        if visual.changed_centroid is not None and visual.changed_count <= 1024:\n            return self._clamp_xy(visual.changed_centroid[0], visual.changed_centroid[1]) + ("changed_centroid",)\n        if visual.centroid is not None:\n            return self._clamp_xy(visual.centroid[0], visual.centroid[1]) + ("occupancy_centroid",)\n        if visual.bbox is not None:\n            center_x = (visual.bbox["x_min"] + visual.bbox["x_max"]) / 2.0\n            center_y = (visual.bbox["y_min"] + visual.bbox["y_max"]) / 2.0\n            return self._clamp_xy(center_x, center_y) + ("bbox_center",)\n        return (GRID_W // 2, GRID_H // 2, "grid_center")\n\n    def _clamp_xy(self, x: float, y: float) -> Tuple[int, int]:\n        return (\n            min(GRID_W - 1, max(0, int(round(x)))),\n            min(GRID_H - 1, max(0, int(round(y)))),\n        )\n\n    def _set_pending(self, action: GameAction, visual: VisualSignature, current_levels: int) -> None:\n        self.pending_action_id = self._action_id(action)\n        self.pending_action_key = visual.key\n        self.pending_before_levels = current_levels\n\n    # ------------------------------------------------------------------\n    # ARC action/state normalization helpers.\n    # ------------------------------------------------------------------\n    def _available_actions(self, latest_frame: FrameData) -> List[GameAction]:\n        raw_available = getattr(latest_frame, "available_actions", None)\n        if raw_available is None:\n            return self._all_non_reset_actions()\n        if isinstance(raw_available, dict):\n            raw_items = list(raw_available.values())\n        elif isinstance(raw_available, (list, tuple, set)):\n            raw_items = list(raw_available)\n        else:\n            raw_items = [raw_available]\n\n        out: List[GameAction] = []\n        seen: set[int] = set()\n        for item in raw_items:\n            action = self._coerce_action(item)\n            if action is None:\n                continue\n            action_id = self._action_id(action)\n            if action_id not in seen:\n                seen.add(action_id)\n                out.append(action)\n        return out\n\n    def _all_non_reset_actions(self) -> List[GameAction]:\n        return [action for action in list(GameAction) if not self._action_is_reset(action)]\n\n    def _coerce_action(self, item: Any) -> Optional[GameAction]:\n        if isinstance(item, GameAction):\n            return item\n        candidate_values: List[Any] = []\n        if isinstance(item, dict):\n            candidate_values.extend([item.get("id"), item.get("value"), item.get("name"), item.get("action")])\n        else:\n            for attr in ("id", "value", "name", "action"):\n                if hasattr(item, attr):\n                    candidate_values.append(getattr(item, attr))\n            candidate_values.append(item)\n\n        for value in candidate_values:\n            if value is None:\n                continue\n            try:\n                if hasattr(GameAction, "from_id") and isinstance(value, (int, float, str)) and str(value).lstrip("-").isdigit():\n                    return GameAction.from_id(int(value))  # type: ignore[attr-defined]\n            except Exception:\n                pass\n            try:\n                if isinstance(value, str) and hasattr(GameAction, value):\n                    return getattr(GameAction, value)\n            except Exception:\n                pass\n            try:\n                for action in list(GameAction):\n                    if value == getattr(action, "value", None) or str(value) == str(getattr(action, "value", "")):\n                        return action\n                    if str(value).upper() == self._action_name(action).upper():\n                        return action\n            except Exception:\n                pass\n        return None\n\n    def _action_id(self, action: GameAction) -> int:\n        for attr in ("id", "value"):\n            if hasattr(action, attr):\n                value = getattr(action, attr)\n                try:\n                    return int(value)\n                except Exception:\n                    try:\n                        return int(str(value).replace("ACTION", ""))\n                    except Exception:\n                        continue\n        name = self._action_name(action)\n        if name == "RESET":\n            return 0\n        if name.startswith("ACTION"):\n            try:\n                return int(name.replace("ACTION", ""))\n            except Exception:\n                pass\n        return int(hashlib.sha256(name.encode("utf-8")).hexdigest()[:4], 16)\n\n    def _action_name(self, action: GameAction) -> str:\n        return str(getattr(action, "name", str(action))).split(".")[-1]\n\n    def _action_is_reset(self, action: GameAction) -> bool:\n        return self._action_name(action).upper() == "RESET" or self._action_id(action) == 0\n\n    def _is_complex_action(self, action: GameAction) -> bool:\n        try:\n            return bool(action.is_complex())  # type: ignore[attr-defined]\n        except Exception:\n            return self._action_id(action) == 6 or self._action_name(action).upper() == "ACTION6"\n\n    def _set_action_data(self, action: GameAction, data: Dict[str, Any]) -> None:\n        try:\n            action.set_data(data)  # type: ignore[attr-defined]\n        except Exception:\n            setattr(action, "action_data", data)\n\n    def _attach_reasoning(self, action: GameAction, reasoning: Any) -> None:\n        try:\n            setattr(action, "reasoning", reasoning)\n        except Exception:\n            pass\n\n    def _state_is(self, state: Any, expected_name: str) -> bool:\n        return self._state_name(state).upper() == expected_name.upper()\n\n    def _state_name(self, state: Any) -> str:\n        if state is None:\n            return "UNKNOWN"\n        if hasattr(state, "name"):\n            return str(getattr(state, "name"))\n        value = getattr(state, "value", state)\n        return str(value).split(".")[-1]\n\n    # ------------------------------------------------------------------\n    # Utility.\n    # ------------------------------------------------------------------\n    def _centroid_tuple(self, points: Iterable[Tuple[int, int]]) -> Optional[Tuple[float, float]]:\n        pts = list(points)\n        if not pts:\n            return None\n        return (\n            round(sum(x for x, _ in pts) / len(pts), 3),\n            round(sum(y for _, y in pts) / len(pts), 3),\n        )\n\n    def _hash_jsonable(self, value: Any) -> str:\n        try:\n            payload = json.dumps(value, sort_keys=True, default=str, separators=(",", ":"))\n        except Exception:\n            payload = repr(value)\n        return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:24]\n\n    def _stable_float(self, seed: str) -> float:\n        value = int(hashlib.sha256(seed.encode("utf-8")).hexdigest()[:12], 16)\n        return value / float(0xFFFFFFFFFFFF)\n\n    def _trace(\n        self,\n        latest_frame: FrameData,\n        visual: VisualSignature,\n        action: GameAction,\n        mode: str,\n        scored: Sequence[Tuple[float, GameAction]],\n    ) -> None:\n        if not self.trace_enabled:\n            return\n        row = {\n            "t": round(time.time(), 6),\n            "game_id": getattr(self, "game_id", getattr(latest_frame, "game_id", "unknown")),\n            "step": self.step_index,\n            "mode": mode,\n            "state": self._state_name(getattr(latest_frame, "state", None)),\n            "levels_completed": int(getattr(latest_frame, "levels_completed", 0) or 0),\n            "action_id": self._action_id(action),\n            "action_name": self._action_name(action),\n            "raw_hash": visual.raw_hash,\n            "braille_hash": visual.braille_hash,\n            "point_count": visual.point_count,\n            "density": visual.density,\n            "bbox": visual.bbox,\n            "centroid": visual.centroid,\n            "changed_count": visual.changed_count,\n            "known_states": len(self.world.known_states),\n            "scores": [\n                [round(score, 5), self._action_id(candidate), self._action_name(candidate)]\n                for score, candidate in list(scored)[:7]\n            ],\n        }\n        try:\n            with open(self.trace_path, "a", encoding="utf-8") as handle:\n                handle.write(json.dumps(row, sort_keys=True) + "\\n")\n        except Exception:\n            pass\n\n\n# Alias for the full ARC-AGI-3-Agents repo when importing by class name.\nBrailleSigil = MyAgent\n'
target_dir = Path("/kaggle/working/agent") if Path("/kaggle/working").exists() else Path("./agent")
target_dir.mkdir(parents=True, exist_ok=True)
target_path = target_dir / "my_agent.py"
target_path.write_text(agent_source, encoding="utf-8")

digest = hashlib.sha256(target_path.read_bytes()).hexdigest()
print("agent/my_agent.py:", target_path)
print("sha256:", digest)
assert digest == "b5017c8758e2cf52c794f79cb06f787cc14a567678cd56c20c10e1be9aaced74"
compile(agent_source, str(target_path), "exec")
print("compile: ok")


In [ ]:
# Local smoke test using fallback classes, then try the Kaggle gateway when present.
import os, sys, importlib, pathlib

sys.path.insert(0, str(pathlib.Path("/kaggle/working").resolve()) if pathlib.Path("/kaggle/working").exists() else str(pathlib.Path(".").resolve()))

from agent.my_agent import MyAgent, FrameData, GameState, ARCGrid

agent = MyAgent()
frame = FrameData(
    frame=[
        [0,0,0,0,0,0,0,0],
        [0,1,1,0,2,2,0,0],
        [0,1,0,0,2,0,0,0],
        [0,0,0,3,3,3,0,0],
        [0,4,0,0,0,0,5,0],
        [0,4,4,0,6,6,5,0],
        [0,0,0,0,0,0,0,0],
    ],
    state=GameState.PLAYING,
    levels_completed=0,
    win_levels=1,
)
action = agent.choose_action([], frame)
desc = ARCGrid.from_matrix(frame.frame).describe()
print("Loaded:", MyAgent)
print("Smoke action:", getattr(action, "name", action))
print("Colors:", desc["colors_present"])
print("Objects:", desc["objects_per_color"])

gateway_candidates = [
    "arc_agi_3.kaggle_gateway",
    "arcagi3.kaggle_gateway",
    "arcengine.kaggle_gateway",
    "arcengine.gateway",
]
ran_gateway = False
for module_name in gateway_candidates:
    try:
        module = importlib.import_module(module_name)
    except Exception:
        continue
    for fn_name in ("run", "main", "run_submission", "run_kaggle"):
        fn = getattr(module, fn_name, None)
        if callable(fn):
            print("Running gateway:", module_name + "." + fn_name)
            try:
                result = fn(MyAgent)
            except TypeError:
                result = fn()
            print("Gateway result:", result)
            ran_gateway = True
            break
    if ran_gateway:
        break

if not ran_gateway:
    print("Gateway module not found in preview. During Kaggle scored rerun, the ARC runtime is expected to execute MyAgent.")


In [ ]:
# Commit-preview fallback only.
# On real Kaggle scoring, the ARC gateway should create submission.parquet.
from pathlib import Path
if not Path("submission.parquet").exists():
    try:
        import pandas as pd
        pd.DataFrame({"status": ["preview_only"], "agent": ["nine18_arc_color_braille_sigil_v2"]}).to_parquet("submission.parquet", index=False)
        print("Created preview submission.parquet placeholder.")
    except Exception as exc:
        print("Could not create preview parquet:", repr(exc))
else:
    print("submission.parquet already exists.")
